# Kev 9B Dataset Collection on Audited COVID Prompts (Base vs. Fine-Tuned, GPU)

## 1. Design

In [ ]:
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import kev_eval_data_audit
import wang_mnl

importlib.reload(wang_mnl)
importlib.reload(kev_eval_data_audit)
from kev_eval_data_audit import (
    build_kev_prompt_items,
    collect_kev_prompts,
    free_kev_memory,
    load_audited_choice_df,
    load_kev_model,
    preview_kev_item,
    verify_matches_gemini_subsample,
)
from wang_mnl import load_covid_conjoint, pair_choice_sets

DEVICE = "cuda"
DTYPE = torch.bfloat16

# Source data and evaluation subsample (same respondents as the Gemini runs)
DATA_PATH = Path("../../data/covid/dataverse_files/Kreps_etal_vax_replication_data.dta")
N_RESPONDENTS = 200
SUBSAMPLE_SEED = 42
GEMINI_REFERENCE = Path("../../proprietary-llms/gemini/data/gemini-3.5-flash-lite_MINIMAL_random.jsonl")
N_QUERY_PERSONAS = None  # None collects all N_RESPONDENTS respondents; a number collects only the first ones

# Output folder (original runs stay in data/)
DATA_DIR = Path("data-audit")

# Respondent persona attributes (columns of data.persona.PERSONA_NAMES), in the order they appear in the state
PERSONA_COLS = ("US_REG", "SEX", "AGE", "EDULEVEL", "RELIGGRP", "PARTY_LR", "US_PRTY", "US_INC", "RACE")

# Persona designs collected for every checkpoint: design name -> state columns
PERSONA_DESIGNS = {
    "random": (),
    "persona": PERSONA_COLS,
}

# Checkpoints: (checkpoint path, model name saved in records, output file slug)
CHECKPOINTS = [
    ("jaredpalmer/kev-9b", "jaredpalmer/kev-9b", "kev-9b-base"),
    ("../../runs/kev-9b-resp-issp+twin2k-grouped-newvalid/best", "kev-9b-resp-issp+twin2k-grouped-newvalid/best", "kev-9b-resp-issp+twin2k-grouped-newvalid_best"),
]
PREVIEW_CHECKPOINT = "jaredpalmer/kev-9b"  # tokenizer used for prompt previews

## 2. Build prompts for each design

In [ ]:
raw_df = load_covid_conjoint(DATA_PATH)
df_A, _, _, _, _ = pair_choice_sets(raw_df)
all_respondents = np.sort(df_A["respondent"].unique())
sampled_respondents = np.sort(np.random.default_rng(SUBSAMPLE_SEED).choice(all_respondents, size=N_RESPONDENTS, replace=False))

choice_df = load_audited_choice_df(df_A)
sub_choice_df = choice_df[choice_df["respondent"].isin(sampled_respondents)].reset_index(drop=True)

design_items = {
    name: build_kev_prompt_items(sub_choice_df, design_name=name, persona_cols=cols)
    for name, cols in PERSONA_DESIGNS.items()
}
for items in design_items.values():
    verify_matches_gemini_subsample(items, gemini_jsonl_path=GEMINI_REFERENCE)

## 3. Inspect designs before collecting

In [ ]:
pd.DataFrame([
    {
        "design": name,
        "output": str(DATA_DIR / f"<slug>_{name}.jsonl"),
        "n state attributes": len(cols),
        "attributes": ", ".join(cols) if cols else "(empty state)",
        "items": len(design_items[name]),
        "respondents": len({it["respondent"] for it in design_items[name]}),
        "distinct states": len({it["state"] for it in design_items[name]}),
    }
    for name, cols in PERSONA_DESIGNS.items()
])

In [ ]:
pd.DataFrame([
    {"attribute": col, "respondents with a value": int(sub_choice_df.drop_duplicates("respondent")[col].notna().sum())}
    for col in PERSONA_COLS
])

In [ ]:
for name in PERSONA_DESIGNS:
    print(f"=== {name} (first item) ===")
    print(preview_kev_item(design_items[name][0], checkpoint=PREVIEW_CHECKPOINT))
    print()

## 4. Collect

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
for ckpt_path, model_name, slug in CHECKPOINTS:
    model = None
    free_kev_memory()
    model = load_kev_model(ckpt_path, device=DEVICE, dtype=DTYPE)
    try:
        for name in PERSONA_DESIGNS:
            records = collect_kev_prompts(
                model,
                design_items[name],
                output_path=DATA_DIR / f"{slug}_{name}.jsonl",
                model_name=model_name,
                n_personas=N_QUERY_PERSONAS,
            )
            print(f"{slug} {name}: mean probs (A, B, C) = {np.round(np.mean([r['probs'] for r in records], axis=0), 4)}")
    finally:
        model = None
        free_kev_memory()